In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# PHASE 1: Data Loading, Validation & Feature Engineering
# Afficionado Coffee Roasters - Sales Trend Analysis
# ============================================================

In [3]:
# ============================================================
# STEP 1: LOAD DATA
# ============================================================
# Place your downloaded CSV in the same folder as this script
# and rename it to coffee_sales.csv
 
df = pd.read_csv("data/coffee_sales.csv")
 
print("=" * 50)
print("STEP 1: RAW DATA OVERVIEW")
print("=" * 50)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nFirst 3 rows:\n{df.head(3)}")

STEP 1: RAW DATA OVERVIEW
Shape: 149116 rows x 11 columns

Column names:
['transaction_id', 'year', 'transaction_time', 'transaction_qty', 'store_id', 'store_location', 'product_id', 'unit_price', 'product_category', 'product_type', 'product_detail']

Data types:
transaction_id        int64
year                  int64
transaction_time        str
transaction_qty       int64
store_id              int64
store_location          str
product_id            int64
unit_price          float64
product_category        str
product_type            str
product_detail          str
dtype: object

First 3 rows:
   transaction_id  year transaction_time  transaction_qty  store_id  \
0               1  2025          7:06:11                2         5   
1               2  2025          7:08:56                2         5   
2               3  2025          7:14:04                2         5   

    store_location  product_id  unit_price    product_category  \
0  Lower Manhattan          32         3.0      

In [4]:
# ============================================================
# STEP 2: VALIDATION CHECKS
# ============================================================
 
print("\n" + "=" * 50)
print("STEP 2: VALIDATION CHECKS")
print("=" * 50)
 
# --- 2a. Missing values ---
missing = df.isnull().sum()
print(f"\nMissing values per column:\n{missing}")
assert missing.sum() == 0, "WARNING: Missing values found! Check above."
print("✓ No missing values.")
 
# --- 2b. Duplicate transaction IDs ---
dup_count = df["transaction_id"].duplicated().sum()
print(f"\nDuplicate transaction_ids: {dup_count}")
assert dup_count == 0, "WARNING: Duplicate transaction_ids found!"
print("✓ No duplicate transaction IDs.")
 
# --- 2c. Sequential check (should run 1 to 149456) ---
expected_ids = set(range(1, df["transaction_id"].max() + 1))
actual_ids   = set(df["transaction_id"])
missing_ids  = expected_ids - actual_ids
print(f"\nExpected ID range: 1 to {df['transaction_id'].max()}")
print(f"Missing IDs in sequence: {len(missing_ids)}")
if missing_ids:
    print(f"  Sample missing IDs: {sorted(missing_ids)[:10]}")
 
# --- 2d. Logical consistency: positive qty and price ---
neg_qty   = (df["transaction_qty"] <= 0).sum()
neg_price = (df["unit_price"] <= 0).sum()
print(f"\nRows with transaction_qty <= 0 : {neg_qty}")
print(f"Rows with unit_price <= 0     : {neg_price}")
assert neg_qty   == 0, "WARNING: Non-positive quantities found!"
assert neg_price == 0, "WARNING: Non-positive prices found!"
print("✓ All quantities and prices are positive.")
 
# --- 2e. year column should be all 2025 ---
unique_years = df["year"].unique()
print(f"\nUnique years in dataset: {unique_years}")
assert list(unique_years) == [2025], "WARNING: Unexpected year values found!"
print("✓ All transactions are from 2025.")
 
# --- 2f. transaction_time format check ---
sample_times = df["transaction_time"].head(5).tolist()
print(f"\nSample transaction_time values: {sample_times}")


STEP 2: VALIDATION CHECKS

Missing values per column:
transaction_id      0
year                0
transaction_time    0
transaction_qty     0
store_id            0
store_location      0
product_id          0
unit_price          0
product_category    0
product_type        0
product_detail      0
dtype: int64
✓ No missing values.

Duplicate transaction_ids: 0
✓ No duplicate transaction IDs.

Expected ID range: 1 to 149456
Missing IDs in sequence: 340
  Sample missing IDs: [3252, 3253, 3254, 3255, 3256, 3257, 3258, 3259, 3260, 3261]

Rows with transaction_qty <= 0 : 0
Rows with unit_price <= 0     : 0
✓ All quantities and prices are positive.

Unique years in dataset: [2025]
✓ All transactions are from 2025.

Sample transaction_time values: ['7:06:11', '7:08:56', '7:14:04', '7:20:24', '7:22:41']


In [5]:
# ============================================================
# STEP 3: PARSE transaction_time
# ============================================================
 
print("\n" + "=" * 50)
print("STEP 3: PARSING TIMESTAMPS")
print("=" * 50)
 
# Handles both '7:06:11' and '07:06:11' formats safely
df["transaction_time_parsed"] = pd.to_datetime(
    df["transaction_time"], format="mixed"
)
 
print(f"Parsed time sample:\n{df['transaction_time_parsed'].head(5)}")
print(f"\nMin time: {df['transaction_time_parsed'].dt.time.min()}")
print(f"Max time: {df['transaction_time_parsed'].dt.time.max()}")
 
# Verify no NaT (failed parses)
nat_count = df["transaction_time_parsed"].isna().sum()
print(f"Failed parses (NaT): {nat_count}")
assert nat_count == 0, "WARNING: Some transaction_time values failed to parse!"
print("✓ All timestamps parsed successfully.")


STEP 3: PARSING TIMESTAMPS
Parsed time sample:
0   2026-06-07 07:06:11
1   2026-06-07 07:08:56
2   2026-06-07 07:14:04
3   2026-06-07 07:20:24
4   2026-06-07 07:22:41
Name: transaction_time_parsed, dtype: datetime64[us]

Min time: 06:00:00
Max time: 20:59:32
Failed parses (NaT): 0
✓ All timestamps parsed successfully.


In [6]:
# ============================================================
# STEP 4: FEATURE ENGINEERING
# ============================================================
 
print("\n" + "=" * 50)
print("STEP 4: FEATURE ENGINEERING")
print("=" * 50)
 
# --- 4a. Revenue per transaction ---
df["revenue"] = df["transaction_qty"] * df["unit_price"]
print(f"\nRevenue column created.")
print(f"  Min: ${df['revenue'].min():.2f}")
print(f"  Max: ${df['revenue'].max():.2f}")
print(f"  Total: ${df['revenue'].sum():,.2f}")
 
# --- 4b. Hour of day (0–23) ---
df["hour"] = df["transaction_time_parsed"].dt.hour
print(f"\nHour range: {df['hour'].min()} to {df['hour'].max()}")
 
# --- 4c. Day of week (0=Monday ... 6=Sunday) as number and label ---
# Using transaction_id as sequential proxy for day ordering
# We bin 149,456 transactions across 365 days (2025)
# Each "day bin" = ~409 transactions
TOTAL_ROWS = len(df)
DAYS_IN_YEAR = 365
 
df = df.sort_values("transaction_id").reset_index(drop=True)
df["day_bin"] = (df.index // (TOTAL_ROWS / DAYS_IN_YEAR)).astype(int)
df["day_bin"] = df["day_bin"].clip(0, DAYS_IN_YEAR - 1)  # safety clip
 
# 2025 starts on Wednesday (weekday=2)
# day_of_week_num: 0=Monday, 6=Sunday
df["day_of_week_num"]  = (df["day_bin"] + 2) % 7    # +2 because Jan 1 2025 = Wednesday
DAY_NAMES = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["day_of_week"] = df["day_of_week_num"].map(lambda x: DAY_NAMES[x])
 
print(f"\nDay of week distribution:")
print(df["day_of_week"].value_counts().reindex(DAY_NAMES))
 
# --- 4d. Week number (1–52) for trend analysis ---
df["week_number"] = (df["day_bin"] // 7) + 1
df["week_number"] = df["week_number"].clip(1, 52)
print(f"\nWeek number range: {df['week_number'].min()} to {df['week_number'].max()}")
 
# --- 4e. Time bucket ---
def assign_time_bucket(hour):
    if   6  <= hour <= 11: return "Morning (6–11)"
    elif 12 <= hour <= 16: return "Afternoon (12–16)"
    elif 17 <= hour <= 21: return "Evening (17–21)"
    else:                  return "Late Hours (22–5)"
 
df["time_bucket"] = df["hour"].apply(assign_time_bucket)
print(f"\nTime bucket distribution:")
print(df["time_bucket"].value_counts())
 
# --- 4f. Is Weekend flag ---
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"])
print(f"\nWeekend transactions : {df['is_weekend'].sum():,}")
print(f"Weekday transactions : {(~df['is_weekend']).sum():,}")


STEP 4: FEATURE ENGINEERING

Revenue column created.
  Min: $0.80
  Max: $360.00
  Total: $698,812.33

Hour range: 6 to 20

Day of week distribution:
day_of_week
Monday       21244
Tuesday      21244
Wednesday    21652
Thursday     21244
Friday       21244
Saturday     21244
Sunday       21244
Name: count, dtype: int64

Week number range: 1 to 52

Time bucket distribution:
time_bucket
Morning (6–11)       81751
Afternoon (12–16)    44427
Evening (17–21)      22938
Name: count, dtype: int64

Weekend transactions : 42,488
Weekday transactions : 106,628


In [7]:
# ============================================================
# STEP 5: FINAL CLEAN DATAFRAME SUMMARY
# ============================================================
 
print("\n" + "=" * 50)
print("STEP 5: FINAL DATAFRAME SUMMARY")
print("=" * 50)
 
# Drop the intermediate parsed time column (keep only what's needed)
df_clean = df.drop(columns=["transaction_time_parsed"])
 
print(f"\nFinal shape   : {df_clean.shape}")
print(f"\nAll columns   : {df_clean.columns.tolist()}")
print(f"\nSample row:\n{df_clean.iloc[0]}")
print(f"\nData types:\n{df_clean.dtypes}")
 
# Quick sanity stats
print("\n--- Revenue Summary ---")
print(f"Total revenue     : ${df_clean['revenue'].sum():>12,.2f}")
print(f"Avg per txn       : ${df_clean['revenue'].mean():>12.2f}")
print(f"Total transactions: {len(df_clean):>12,}")
 
print("\n--- Store Locations ---")
print(df_clean["store_location"].value_counts())
 
print("\n--- Product Categories ---")
print(df_clean["product_category"].value_counts())


STEP 5: FINAL DATAFRAME SUMMARY

Final shape   : (149116, 19)

All columns   : ['transaction_id', 'year', 'transaction_time', 'transaction_qty', 'store_id', 'store_location', 'product_id', 'unit_price', 'product_category', 'product_type', 'product_detail', 'revenue', 'hour', 'day_bin', 'day_of_week_num', 'day_of_week', 'week_number', 'time_bucket', 'is_weekend']

Sample row:
transaction_id                          1
year                                 2025
transaction_time                  7:06:11
transaction_qty                         2
store_id                                5
store_location            Lower Manhattan
product_id                             32
unit_price                            3.0
product_category                   Coffee
product_type        Gourmet brewed coffee
product_detail                Ethiopia Rg
revenue                               6.0
hour                                    7
day_bin                                 0
day_of_week_num                  

In [8]:
# ============================================================
# STEP 6: SAVE CLEAN DATA
# ============================================================
 
df_clean.to_csv("data/coffee_sales_clean.csv", index=False)
print("\n" + "=" * 50)
print("✓ PHASE 1 COMPLETE")
print("  Saved: data/coffee_sales_clean.csv")
print("  Rows  :", len(df_clean))
print("  Cols  :", len(df_clean.columns))
print("=" * 50)


✓ PHASE 1 COMPLETE
  Saved: data/coffee_sales_clean.csv
  Rows  : 149116
  Cols  : 19


# ============================================================
# PHASE 2: Analysis & Visualizations
# Afficionado Coffee Roasters - Sales Trend Analysis
# =========

In [9]:
# ============================================================
# LOAD CLEAN DATA
# ============================================================
 
df = pd.read_csv("data/coffee_sales_clean.csv")
 
# Restore ordered categories for correct chart sorting
DAY_ORDER    = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
BUCKET_ORDER = ["Morning (6–11)", "Afternoon (12–16)", "Evening (17–21)"]
LOCATIONS    = df["store_location"].unique().tolist()
 
df["day_of_week"] = pd.Categorical(df["day_of_week"], categories=DAY_ORDER, ordered=True)
df["time_bucket"] = pd.Categorical(df["time_bucket"], categories=BUCKET_ORDER, ordered=True)
 
print("Data loaded:", df.shape)

Data loaded: (149116, 19)


In [11]:
# ============================================================
# CHART 1: Weekly Sales Trend (Revenue + Transaction Count)
# ============================================================
 
weekly = (
    df.groupby("week_number")
    .agg(total_revenue=("revenue", "sum"),
         total_transactions=("transaction_id", "count"))
    .reset_index()
)
 
fig1 = make_subplots(specs=[[{"secondary_y": True}]])
 
fig1.add_trace(
    go.Scatter(
        x=weekly["week_number"],
        y=weekly["total_revenue"].round(2),
        name="Revenue ($)",
        line=dict(color="#1D9E75", width=2.5),
        fill="tozeroy",
        fillcolor="rgba(29,158,117,0.08)",
        mode="lines"
    ),
    secondary_y=False
)
 
fig1.add_trace(
    go.Scatter(
        x=weekly["week_number"],
        y=weekly["total_transactions"],
        name="Transactions",
        line=dict(color="#534AB7", width=2, dash="dot"),
        mode="lines"
    ),
    secondary_y=True
)
 
# Trend line for revenue
z = np.polyfit(weekly["week_number"], weekly["total_revenue"], 1)
p = np.poly1d(z)
fig1.add_trace(
    go.Scatter(
        x=weekly["week_number"],
        y=p(weekly["week_number"]).round(2),
        name="Revenue Trend",
        line=dict(color="#D85A30", width=1.5, dash="dash"),
        mode="lines"
    ),
    secondary_y=False
)
 
fig1.update_layout(
    title="Weekly Sales Trend Across 2025",
    xaxis_title="Week Number",
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified",
    font=dict(family="Arial", size=13)
)
fig1.update_yaxes(title_text="Revenue ($)", secondary_y=False, gridcolor="#f0f0f0")
fig1.update_yaxes(title_text="Transaction Count", secondary_y=True, gridcolor="#f0f0f0")
 
fig1.write_html("charts/chart1_weekly_trend.html")
fig1.show()
print("✓ Chart 1: Weekly Sales Trend saved.")

✓ Chart 1: Weekly Sales Trend saved.


In [12]:
# ============================================================
# CHART 2: Day-of-Week Performance (Avg Revenue + Avg Transactions)
# ============================================================
 
dow = (
    df.groupby("day_of_week", observed=True)
    .agg(avg_revenue=("revenue", "mean"),
         total_revenue=("revenue", "sum"),
         avg_transactions=("transaction_id", "count"))
    .reset_index()
)
# avg_transactions = total per day-of-week / number of weeks
dow["avg_transactions"] = (dow["avg_transactions"] / 52).round(1)
dow["avg_revenue"] = dow["avg_revenue"].round(2)
 
fig2 = make_subplots(rows=1, cols=2,
                     subplot_titles=("Avg Revenue per Transaction by Day",
                                     "Avg Daily Transaction Count by Day"))
 
colors_dow = ["#534AB7" if d not in ["Saturday","Sunday"] else "#D85A30"
              for d in dow["day_of_week"].tolist()]
 
fig2.add_trace(
    go.Bar(
        x=dow["day_of_week"].tolist(),
        y=dow["avg_revenue"],
        marker_color=colors_dow,
        name="Avg Revenue",
        text=["$" + str(v) for v in dow["avg_revenue"]],
        textposition="outside"
    ),
    row=1, col=1
)
 
fig2.add_trace(
    go.Bar(
        x=dow["day_of_week"].tolist(),
        y=dow["avg_transactions"],
        marker_color=colors_dow,
        name="Avg Transactions",
        text=dow["avg_transactions"].astype(str),
        textposition="outside"
    ),
    row=1, col=2
)
 
fig2.update_layout(
    title="Day-of-Week Performance Analysis<br><sup>Purple = Weekday | Orange = Weekend</sup>",
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    font=dict(family="Arial", size=13)
)
fig2.update_yaxes(gridcolor="#f0f0f0")
 
fig2.write_html("charts/chart2_day_of_week.html")
fig2.show()
print("✓ Chart 2: Day-of-Week Performance saved.")

✓ Chart 2: Day-of-Week Performance saved.


In [13]:
# ============================================================
# CHART 3: Weekday vs Weekend Comparison
# ============================================================
 
weekend_comp = (
    df.groupby("is_weekend")
    .agg(total_revenue=("revenue", "sum"),
         total_transactions=("transaction_id", "count"),
         avg_revenue_per_txn=("revenue", "mean"))
    .reset_index()
)
weekend_comp["label"] = weekend_comp["is_weekend"].map({True: "Weekend", False: "Weekday"})
weekend_comp["avg_revenue_per_txn"] = weekend_comp["avg_revenue_per_txn"].round(2)
weekend_comp["total_revenue"] = weekend_comp["total_revenue"].round(2)
 
fig3 = make_subplots(rows=1, cols=3,
                     subplot_titles=("Total Revenue", "Total Transactions", "Avg Revenue/Transaction"))
 
metrics = ["total_revenue", "total_transactions", "avg_revenue_per_txn"]
colors_we = ["#534AB7", "#D85A30"]
 
for i, m in enumerate(metrics, 1):
    fig3.add_trace(
        go.Bar(
            x=weekend_comp["label"].tolist(),
            y=weekend_comp[m],
            marker_color=colors_we,
            text=weekend_comp[m].round(2).astype(str),
            textposition="outside",
            showlegend=False
        ),
        row=1, col=i
    )
 
fig3.update_layout(
    title="Weekday vs Weekend Comparison",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=13)
)
fig3.update_yaxes(gridcolor="#f0f0f0")
 
fig3.write_html("charts/chart3_weekday_vs_weekend.html")
fig3.show()
print("✓ Chart 3: Weekday vs Weekend saved.")

✓ Chart 3: Weekday vs Weekend saved.


In [14]:
# ============================================================
# CHART 4: Hourly Demand Curve (All Stores Combined)
# ============================================================
 
hourly = (
    df.groupby("hour")
    .agg(total_transactions=("transaction_id", "count"),
         total_revenue=("revenue", "sum"))
    .reset_index()
)
hourly["total_revenue"] = hourly["total_revenue"].round(2)
 
fig4 = make_subplots(specs=[[{"secondary_y": True}]])
 
fig4.add_trace(
    go.Scatter(
        x=hourly["hour"],
        y=hourly["total_transactions"],
        name="Transactions",
        line=dict(color="#534AB7", width=3),
        fill="tozeroy",
        fillcolor="rgba(83,74,183,0.08)",
        mode="lines+markers",
        marker=dict(size=7)
    ),
    secondary_y=False
)
 
fig4.add_trace(
    go.Scatter(
        x=hourly["hour"],
        y=hourly["total_revenue"],
        name="Revenue ($)",
        line=dict(color="#1D9E75", width=2.5, dash="dot"),
        mode="lines+markers",
        marker=dict(size=6)
    ),
    secondary_y=True
)
 
# Annotate peak hour
peak_hour = hourly.loc[hourly["total_transactions"].idxmax(), "hour"]
peak_txns = hourly.loc[hourly["total_transactions"].idxmax(), "total_transactions"]
 
fig4.add_annotation(
    x=peak_hour, y=peak_txns,
    text=f"Peak: {peak_hour}:00",
    showarrow=True, arrowhead=2,
    arrowcolor="#D85A30", font=dict(color="#D85A30", size=12),
    yshift=15
)
 
fig4.update_layout(
    title="Hourly Transaction & Revenue Distribution (All Stores)",
    xaxis=dict(title="Hour of Day", tickmode="linear", tick0=6, dtick=1),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified",
    font=dict(family="Arial", size=13)
)
fig4.update_yaxes(title_text="Transaction Count", secondary_y=False, gridcolor="#f0f0f0")
fig4.update_yaxes(title_text="Revenue ($)", secondary_y=True, gridcolor="#f0f0f0")
 
fig4.write_html("charts/chart4_hourly_demand.html")
fig4.show()
print("✓ Chart 4: Hourly Demand Curve saved.")

✓ Chart 4: Hourly Demand Curve saved.


In [15]:
# ============================================================
# CHART 5: Hourly Heatmap per Store Location
# ============================================================
 
heatmap_data = (
    df.groupby(["store_location", "hour"])["transaction_id"]
    .count()
    .reset_index(name="transactions")
)
 
# Pivot: rows = store, cols = hour
heatmap_pivot = heatmap_data.pivot(index="store_location", columns="hour", values="transactions").fillna(0)
 
fig5 = go.Figure(data=go.Heatmap(
    z=heatmap_pivot.values,
    x=[f"{h}:00" for h in heatmap_pivot.columns],
    y=heatmap_pivot.index.tolist(),
    colorscale="Teal",
    text=heatmap_pivot.values.astype(int),
    texttemplate="%{text}",
    hovertemplate="Store: %{y}<br>Hour: %{x}<br>Transactions: %{z}<extra></extra>",
    colorbar=dict(title="Transactions")
))
 
fig5.update_layout(
    title="Transaction Volume Heatmap by Store and Hour",
    xaxis_title="Hour of Day",
    yaxis_title="Store Location",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=13),
    height=350
)
 
fig5.write_html("charts/chart5_heatmap_store_hour.html")
fig5.show()
print("✓ Chart 5: Store-Hour Heatmap saved.")

✓ Chart 5: Store-Hour Heatmap saved.


In [16]:
# ============================================================
# CHART 6: Time Bucket Distribution by Store
# ============================================================
 
bucket_store = (
    df.groupby(["store_location", "time_bucket"], observed=True)["transaction_id"]
    .count()
    .reset_index(name="transactions")
)
 
fig6 = px.bar(
    bucket_store,
    x="store_location",
    y="transactions",
    color="time_bucket",
    barmode="group",
    title="Transaction Count by Time Bucket and Store Location",
    labels={"transactions": "Transaction Count",
            "store_location": "Store Location",
            "time_bucket": "Time Bucket"},
    color_discrete_map={
        "Morning (6–11)":     "#534AB7",
        "Afternoon (12–16)":  "#1D9E75",
        "Evening (17–21)":    "#D85A30"
    },
    category_orders={"time_bucket": BUCKET_ORDER}
)
 
fig6.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=13),
    legend_title="Time Bucket"
)
fig6.update_yaxes(gridcolor="#f0f0f0")
 
fig6.write_html("charts/chart6_bucket_by_store.html")
fig6.show()
print("✓ Chart 6: Time Bucket by Store saved.")

✓ Chart 6: Time Bucket by Store saved.


In [17]:
# ============================================================
# CHART 7: Weekly Revenue Trend per Store (Comparison)
# ============================================================
 
weekly_store = (
    df.groupby(["store_location", "week_number"])["revenue"]
    .sum()
    .reset_index()
)
weekly_store["revenue"] = weekly_store["revenue"].round(2)
 
fig7 = px.line(
    weekly_store,
    x="week_number",
    y="revenue",
    color="store_location",
    title="Weekly Revenue Trend by Store Location",
    labels={"revenue": "Revenue ($)", "week_number": "Week Number",
            "store_location": "Store Location"},
    color_discrete_map={
        "Hell's Kitchen":   "#534AB7",
        "Astoria":          "#1D9E75",
        "Lower Manhattan":  "#D85A30"
    }
)
 
fig7.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=13),
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig7.update_yaxes(gridcolor="#f0f0f0")
fig7.update_traces(line_width=2)
 
fig7.write_html("charts/chart7_weekly_by_store.html")
fig7.show()
print("✓ Chart 7: Weekly Revenue by Store saved.")

✓ Chart 7: Weekly Revenue by Store saved.


In [18]:
# ============================================================
# SUMMARY STATS TABLE (for Research Paper)
# ============================================================
 
print("\n" + "=" * 60)
print("SUMMARY STATISTICS FOR RESEARCH PAPER")
print("=" * 60)
 
print("\n--- Weekly Revenue Stats ---")
print(f"  Average weekly revenue : ${weekly['total_revenue'].mean():>10,.2f}")
print(f"  Peak week (week no.)   : Week {weekly.loc[weekly['total_revenue'].idxmax(),'week_number']}")
print(f"  Peak week revenue      : ${weekly['total_revenue'].max():>10,.2f}")
print(f"  Lowest week revenue    : ${weekly['total_revenue'].min():>10,.2f}")
 
print("\n--- Day-of-Week Stats ---")
best_day  = dow.loc[dow["avg_transactions"].idxmax(), "day_of_week"]
worst_day = dow.loc[dow["avg_transactions"].idxmin(), "day_of_week"]
print(f"  Busiest day (avg txns) : {best_day}")
print(f"  Slowest day (avg txns) : {worst_day}")
 
print("\n--- Hourly Stats ---")
print(f"  Peak hour              : {peak_hour}:00")
print(f"  Peak hour transactions : {peak_txns:,}")
slowest_hour = hourly.loc[hourly["total_transactions"].idxmin(), "hour"]
print(f"  Slowest hour           : {slowest_hour}:00")
 
print("\n--- Time Bucket Stats ---")
print(df.groupby("time_bucket", observed=True)["revenue"]
      .agg(["sum","mean","count"])
      .rename(columns={"sum":"Total Revenue","mean":"Avg Revenue","count":"Transactions"})
      .round(2))
 
print("\n--- Store Comparison ---")
store_summary = (
    df.groupby("store_location")
    .agg(total_revenue=("revenue","sum"),
         total_txns=("transaction_id","count"),
         avg_txn_value=("revenue","mean"))
    .round(2)
)
print(store_summary)
 
print("\n✓ PHASE 2 COMPLETE — All 7 charts saved to charts/ folder")


SUMMARY STATISTICS FOR RESEARCH PAPER

--- Weekly Revenue Stats ---
  Average weekly revenue : $ 13,438.70
  Peak week (week no.)   : Week 52
  Peak week revenue      : $ 15,069.22
  Lowest week revenue    : $ 12,578.70

--- Day-of-Week Stats ---
  Busiest day (avg txns) : Wednesday
  Slowest day (avg txns) : Monday

--- Hourly Stats ---
  Peak hour              : 10:00
  Peak hour transactions : 18,545
  Slowest hour           : 20:00

--- Time Bucket Stats ---
                   Total Revenue  Avg Revenue  Transactions
time_bucket                                                
Morning (6–11)         388288.67         4.75         81751
Afternoon (12–16)      204720.83         4.61         44427
Evening (17–21)        105802.83         4.61         22938

--- Store Comparison ---
                 total_revenue  total_txns  avg_txn_value
store_location                                           
Astoria              232243.91       50599           4.59
Hell's Kitchen       236511.17  